# Análise exploratória do dataset sintético de churn

Esta análise tem como objetivo entender a estrutura do dataset sintético, verificar a qualidade dos dados e identificar os primeiros sinais associados ao churn voluntário em clientes de um provedor de internet.

Os dados usados aqui são sintéticos. As taxas observadas nesta análise servem para validar a coerência do projeto e não representam resultados reais da empresa.

In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH_CANDIDATES = [
    Path("data/customers_churn_synthetic.csv"),
    Path("../data/customers_churn_synthetic.csv"),
]

DATA_PATH = next(path for path in DATA_PATH_CANDIDATES if path.exists())

df = pd.read_csv(DATA_PATH)

df.head()


,customer_id,tenure_months,access_technology,download_speed_mbps,monthly_fee,has_contract_loyalty,overdue_invoice_count,oldest_overdue_days,active_agreement_installment_amount,had_price_adjustment_90d,support_tickets_90d,repeat_issue_90d,avg_resolution_hours_90d,outage_count_30d,network_outage_hours_30d,churn_90d
0,CUST-000001,40,fiber,500,160.93,0,0,0,0.00,1,0,0,0.00,0,0.00,0
1,CUST-000002,53,fiber,200,109.99,1,0,0,0.00,0,0,0,0.00,0,0.00,0
2,CUST-000003,35,fiber,100,100.73,0,1,46,0.00,0,0,0,0.00,0,0.00,0
3,CUST-000004,31,fiber,200,116.91,1,0,0,61.27,0,0,0,0.00,1,1.13,1
4,CUST-000005,58,fiber,100,105.92,1,2,32,0.00,0,1,0,47.23,0,0.00,0


In [2]:
print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

df.dtypes

Linhas: 5000
Colunas: 16


customer_id                             object
tenure_months                            int64
access_technology                       object
download_speed_mbps                      int64
monthly_fee                            float64
has_contract_loyalty                     int64
overdue_invoice_count                    int64
oldest_overdue_days                      int64
active_agreement_installment_amount    float64
had_price_adjustment_90d                 int64
support_tickets_90d                      int64
repeat_issue_90d                         int64
avg_resolution_hours_90d               float64
outage_count_30d                         int64
network_outage_hours_30d               float64
churn_90d                                int64
dtype: object

In [3]:
missing_values = df.isna().sum()
duplicated_customers = df["customer_id"].duplicated().sum()

print(f"Total de valores nulos: {missing_values.sum()}")
print(f"Clientes duplicados: {duplicated_customers}")

missing_values

Total de valores nulos: 0
Clientes duplicados: 0


customer_id                            0
tenure_months                          0
access_technology                      0
download_speed_mbps                    0
monthly_fee                            0
has_contract_loyalty                   0
overdue_invoice_count                  0
oldest_overdue_days                    0
active_agreement_installment_amount    0
had_price_adjustment_90d               0
support_tickets_90d                    0
repeat_issue_90d                       0
avg_resolution_hours_90d               0
outage_count_30d                       0
network_outage_hours_30d               0
churn_90d                              0
dtype: int64

In [4]:
checks = {
    "oldest_overdue_zero_when_no_overdue_invoice": (
        df.loc[df["overdue_invoice_count"] == 0, "oldest_overdue_days"] == 0
    ).all(),
    "overdue_invoice_positive_when_oldest_overdue_positive": (
        df.loc[df["oldest_overdue_days"] > 0, "overdue_invoice_count"] > 0
    ).all(),
    "agreement_can_exist_without_overdue_invoice": (
        (df["active_agreement_installment_amount"] > 0)
        & (df["overdue_invoice_count"] == 0)
    ).any(),
    "resolution_zero_when_no_support_ticket": (
        df.loc[df["support_tickets_90d"] == 0, "avg_resolution_hours_90d"] == 0
    ).all(),
    "outage_hours_zero_when_no_outage": (
        df.loc[df["outage_count_30d"] == 0, "network_outage_hours_30d"] == 0
    ).all(),
}

checks

{'oldest_overdue_zero_when_no_overdue_invoice': np.True_,
 'overdue_invoice_positive_when_oldest_overdue_positive': np.True_,
 'agreement_can_exist_without_overdue_invoice': np.True_,
 'resolution_zero_when_no_support_ticket': np.True_,
 'outage_hours_zero_when_no_outage': np.True_}

In [5]:
churn_rate = df["churn_90d"].mean()
overdue_rate = (df["overdue_invoice_count"] > 0).mean()
agreement_rate = (df["active_agreement_installment_amount"] > 0).mean()

print(f"Taxa de churn voluntário: {churn_rate:.2%}")
print(f"Clientes com fatura vencida: {overdue_rate:.2%}")
print(f"Clientes com acordo ativo: {agreement_rate:.2%}")

Taxa de churn voluntário: 10.34%
Clientes com fatura vencida: 24.24%
Clientes com acordo ativo: 10.78%


In [6]:
technology_distribution = df["access_technology"].value_counts(normalize=True).mul(100).round(2)
loyalty_distribution = df["has_contract_loyalty"].value_counts(normalize=True).mul(100).round(2)

print("Distribuição por tecnologia de acesso (%):")
display(technology_distribution)

print("Distribuição por fidelidade (%):")
display(loyalty_distribution)

Distribuição por tecnologia de acesso (%):


access_technology
fiber    75.9
radio    24.1
Name: proportion, dtype: float64

Distribuição por fidelidade (%):


has_contract_loyalty
1    61.06
0    38.94
Name: proportion, dtype: float64

In [7]:
def count_and_percent(column):
    counts = df[column].value_counts().rename("clientes")
    percents = df[column].value_counts(normalize=True).mul(100).round(2).rename("percentual")
    
    return pd.concat([counts, percents], axis=1)


count_and_percent("access_technology")

,clientes,percentual
access_technology,,
fiber,3795,75.9
radio,1205,24.1


In [8]:
categorical_columns = [
    "access_technology",
    "has_contract_loyalty",
    "had_price_adjustment_90d",
    "repeat_issue_90d",
    "churn_90d",
]

for column in categorical_columns:
    print(f"\nDistribuição de {column}:")
    display(count_and_percent(column))


Distribuição de access_technology:


,clientes,percentual
access_technology,,
fiber,3795,75.9
radio,1205,24.1



Distribuição de has_contract_loyalty:


,clientes,percentual
has_contract_loyalty,,
1,3053,61.06
0,1947,38.94



Distribuição de had_price_adjustment_90d:


,clientes,percentual
had_price_adjustment_90d,,
0,4079,81.58
1,921,18.42



Distribuição de repeat_issue_90d:


,clientes,percentual
repeat_issue_90d,,
0,4888,97.76
1,112,2.24



Distribuição de churn_90d:


,clientes,percentual
churn_90d,,
0,4483,89.66
1,517,10.34


In [9]:
numeric_columns = [
    "tenure_months",
    "download_speed_mbps",
    "monthly_fee",
    "overdue_invoice_count",
    "oldest_overdue_days",
    "active_agreement_installment_amount",
    "support_tickets_90d",
    "avg_resolution_hours_90d",
    "outage_count_30d",
    "network_outage_hours_30d",
]

df[numeric_columns].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).T

,count,mean,std,min,25%,50%,75%,95%,max
tenure_months,5000.0,37.659800,24.926523,1.0,19.0000,32.000,50.2500,88.0000,120.00
download_speed_mbps,5000.0,237.412000,177.649357,20.0,100.0000,200.000,300.0000,600.0000,600.00
monthly_fee,5000.0,116.841910,30.952338,59.9,92.8375,113.405,133.9675,172.5905,195.95
overdue_invoice_count,5000.0,0.310800,0.623122,0.0,0.0000,0.000,0.0000,2.0000,5.00
oldest_overdue_days,5000.0,11.926800,26.655190,0.0,0.0000,0.000,0.0000,73.0000,180.00
active_agreement_installment_amount,5000.0,6.031714,19.367338,0.0,0.0000,0.000,0.0000,52.9275,142.56
support_tickets_90d,5000.0,0.431800,0.673676,0.0,0.0000,0.000,1.0000,2.0000,6.00
avg_resolution_hours_90d,5000.0,8.056812,13.716006,0.0,0.0000,0.000,13.9325,37.1625,104.56
outage_count_30d,5000.0,0.676400,0.895680,0.0,0.0000,0.000,1.0000,2.0000,7.00
network_outage_hours_30d,5000.0,1.368080,2.616421,0.0,0.0000,0.000,1.8700,6.1210,51.90


## Análise de churn por grupos

Depois das validações iniciais, a próxima etapa é comparar a taxa de churn voluntário entre grupos de clientes. O objetivo é identificar associações relevantes, sem interpretar essas diferenças como causalidade.

In [10]:
def churn_by_group(column):
    summary = (
        df.groupby(column, observed=True)
        .agg(
            clientes=("customer_id", "count"),
            churns=("churn_90d", "sum"),
            taxa_churn=("churn_90d", "mean"),
        )
        .sort_index()
    )
    
    summary["taxa_churn"] = summary["taxa_churn"].mul(100).round(2)
    
    return summary


churn_by_group("access_technology")

,clientes,churns,taxa_churn
access_technology,,,
fiber,3795,377,9.93
radio,1205,140,11.62


In [11]:
binary_group_columns = [
    "has_contract_loyalty",
    "had_price_adjustment_90d",
    "repeat_issue_90d",
]

for column in binary_group_columns:
    print(f"\nChurn por {column}:")
    display(churn_by_group(column))


Churn por has_contract_loyalty:


,clientes,churns,taxa_churn
has_contract_loyalty,,,
0,1947,250,12.84
1,3053,267,8.75



Churn por had_price_adjustment_90d:


,clientes,churns,taxa_churn
had_price_adjustment_90d,,,
0,4079,396,9.71
1,921,121,13.14



Churn por repeat_issue_90d:


,clientes,churns,taxa_churn
repeat_issue_90d,,,
0,4888,490,10.02
1,112,27,24.11


In [12]:
df["overdue_invoice_count_band"] = pd.cut(
    df["overdue_invoice_count"],
    bins=[-1, 0, 1, 2, 8],
    labels=["0", "1", "2", "3+"],
)

churn_by_group("overdue_invoice_count_band")


,clientes,churns,taxa_churn
overdue_invoice_count_band,,,
0,3788,363,9.58
1,945,108,11.43
2,208,35,16.83
3+,59,11,18.64


In [13]:
df["has_active_agreement"] = df["active_agreement_installment_amount"] > 0

churn_by_group("has_active_agreement")

,clientes,churns,taxa_churn
has_active_agreement,,,
False,4461,459,10.29
True,539,58,10.76


In [14]:
churn_by_group("support_tickets_90d")

,clientes,churns,taxa_churn
support_tickets_90d,,,
0,3269,279,8.53
1,1370,166,12.12
2,311,57,18.33
3,40,9,22.50
4,5,3,60.00
5,3,1,33.33
6,2,2,100.00


In [15]:
df["support_tickets_band"] = pd.cut(
    df["support_tickets_90d"],
    bins=[-1, 0, 1, 2, 6],
    labels=["0", "1", "2", "3+"],
)

churn_by_group("support_tickets_band")

,clientes,churns,taxa_churn
support_tickets_band,,,
0,3269,279,8.53
1,1370,166,12.12
2,311,57,18.33
3+,50,15,30.00


In [16]:
df["network_outage_hours_band"] = pd.cut(
    df["network_outage_hours_30d"],
    bins=[-1, 0, 2, 6, 12, 72],
    labels=["0", "0-2", "2-6", "6-12", "12+"],
)

churn_by_group("network_outage_hours_band")

,clientes,churns,taxa_churn
network_outage_hours_band,,,
0,2681,248,9.25
0-2,1152,113,9.81
2-6,905,116,12.82
6-12,222,33,14.86
12+,40,7,17.50


### Leituras parciais

Até aqui, a base sintética parece consistente com o dicionário de dados: não há nulos, não há clientes duplicados e as principais regras financeiras e operacionais foram respeitadas.

A taxa geral de churn voluntário ficou em torno de 10%, dentro da faixa definida para a V1. A inadimplência simulada ficou próxima de 25%, mas essa proporção é uma hipótese do dataset sintético e não representa taxa real da empresa.

Os primeiros cruzamentos indicam churn maior entre clientes sem fidelidade, com reajuste recente, com reincidência de problemas, com mais faturas vencidas, com mais chamados e com mais horas de indisponibilidade. Essas diferenças indicam associação, não causalidade.